# Inspect fixed-width control-prefix tokenization

Verify random no-gate data, both one-token gate values, and constant prefix length across every bitstring of the configured width; then print token boundaries.

In [1]:
import random
import re
import sys
from itertools import product
from pathlib import Path

import torch
from transformers import AutoTokenizer

start = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()
repo_root = next(path for path in [start, *start.parents] if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(repo_root))
from ciphers.kirchenbauer_et_al.binary_classification_mvp.data import compile_prefix, load_fineweb, prefix_batch, tokenize_with_prefix  # noqa: E402

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Base")
N_BITS = 8
gate_pattern = re.compile(r"<do_encoding> (yes|no) </do_encoding>")
bits_pattern = re.compile(r"<encoding_value> ([01]+) </encoding_value>")

/mnt/align4_drive2/adrianoh/miniconda-installation/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rows = list(load_fineweb(n=10))
texts, sampled_bits, gates = prefix_batch([row["text"] for row in rows], N_BITS, do_encoding=False)
assert gates == [False] * len(rows)
assert all(gate_pattern.search(text).group(1) == "no" for text in texts)
assert all(len(bits) == N_BITS and set(bits) <= {"0", "1"} for bits in sampled_bits)
assert len(set(sampled_bits)) > 1
random.seed(42)
_, random_bits, random_gates = prefix_batch([""] * 10, N_BITS)
assert set(random_gates) == {False, True} and all(len(bits) == N_BITS for bits in random_bits)
print("Random no-gate bitstrings:", sampled_bits)

Random no-gate bitstrings: ['01000011', '00110000', '11001110', '11011101', '10010100', '01110110', '10111111', '01110111', '01100010', '00010101']


In [3]:
prefix_lengths = set()
for enabled, values in product((False, True), product("01", repeat=N_BITS)):
    prefix = compile_prefix("".join(values), enabled)
    encoded = tokenizer(prefix, add_special_tokens=False, return_offsets_mapping=True)
    prefix_lengths.add(len(encoded["input_ids"]))
    gate_start, gate_end = gate_pattern.search(prefix).span(1)
    gate_tokens = [(s, e) for s, e in encoded["offset_mapping"] if s < gate_end and e > gate_start]
    assert len(gate_tokens) == 1

assert len(prefix_lengths) == 1
print(f"Verified all {2 * (1 << N_BITS)} prefixes have {prefix_lengths.pop()} tokens and one-token gates.")

Verified all 512 prefixes have 32 tokens and one-token gates.


In [4]:
for enabled in (False, True):
    prefix = compile_prefix("01001101", enabled)
    encoded = tokenizer(prefix, add_special_tokens=False, return_offsets_mapping=True)
    start, end = bits_pattern.search(prefix).span(1)
    boundaries = [(max(s, start) - start, min(e, end) - start, prefix[max(s, start) : min(e, end)]) for s, e in encoded["offset_mapping"] if s < end and e > start]
    print(f"gate={gate_pattern.search(prefix).group(1)} bits={prefix[start:end]} token_boundaries={boundaries}")

gate=no bits=01001101 token_boundaries=[(0, 1, '0'), (1, 2, '1'), (2, 3, '0'), (3, 4, '0'), (4, 5, '1'), (5, 6, '1'), (6, 7, '0'), (7, 8, '1')]
gate=yes bits=01001101 token_boundaries=[(0, 1, '0'), (1, 2, '1'), (2, 3, '0'), (3, 4, '0'), (4, 5, '1'), (5, 6, '1'), (6, 7, '0'), (7, 8, '1')]


In [5]:
comparison_rows = list(load_fineweb(n=100))
for n_bits in range(1, N_BITS + 1):
    messages = ["".join(values) for values in product("01", repeat=n_bits)]
    bits = messages * 2
    gates = [False] * len(messages) + [True] * len(messages)
    for row in comparison_rows:
        texts = [row["text"]] * len(bits)
        token_space, _, _ = tokenize_with_prefix(tokenizer, texts, bits, gates, 64, "token")
        character_space, _, _ = tokenize_with_prefix(tokenizer, texts, bits, gates, 64, "character")
        assert all(torch.equal(token_space[key], character_space[key]) for key in ("input_ids", "attention_mask"))

print(f"Token- and character-space concatenation match for 100 texts and all messages up to {N_BITS} bits.")

Token- and character-space concatenation match for 100 texts and all messages up to 8 bits.
